# Box-Jenkins Zeitreihenanalyse: Gold (GC=F)
> **Projekt:** Asset Forecasting — DAX, Gold & Bitcoin  
> **Kurs:** Vertiefung Business Analytics · THWS · SoSe 2026  
> **Prof.:** Dr. Christian Menden  
> **Autor:** Antonio Sicaja  
> **Asset:** Gold (Ticker: `GC=F`) · täglich · 2016–2026

---

## Methodik — Box-Jenkins in 4 Schritten

1. **Identifikation** — Stationaritätsprüfung (ADF, KPSS), ACF/PACF
2. **Schätzung** — Gittersuche über ARIMA(p,d,q)
3. **Diagnostik** — Residualanalyse (Ljung-Box, Jarque-Bera)
4. **Prognose** — 10-Perioden-Forecast mit 95%-Konfidenzintervall

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
import logging
logging.captureWarnings(True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.style.use('seaborn-v0_8-darkgrid')
GOLD  = '#FFD700'
BLUE  = '#2196F3'
RED   = '#F44336'
GREEN = '#4CAF50'

print('Setup abgeschlossen ✓')

---
## 1. Daten laden & aufbereiten

In [ ]:
DATA_PATH = '../data/raw/goldpreis_2016_2026.csv'

df = pd.read_csv(DATA_PATH, skiprows=1)
df.columns = ['Date', 'Close']
df = df.dropna()
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
close = df['Close'].dropna()

log_ret   = np.log(close).diff().dropna()
log_close = np.log(close.values)  # numpy array → keine Index-Warnungen

print(f'Zeitraum    : {close.index[0].date()} → {close.index[-1].date()}')
print(f'Handelstage : {len(close)}')
print(f'Preis Start : {close.iloc[0]:.2f} USD/oz')
print(f'Preis Ende  : {close.iloc[-1]:.2f} USD/oz')
print(f'Gesamtrendite: {(close.iloc[-1]/close.iloc[0] - 1)*100:.1f}%')

In [ ]:
desc = pd.DataFrame({
    'Preis (USD/oz)': close.describe(),
    'Log-Returns': log_ret.describe()
})
print(desc.round(6))
print(f'\nLog-Returns:')
print(f'  Schiefe  : {log_ret.skew():.4f}')
print(f'  Kurtosis : {log_ret.kurtosis():.4f}  (Normal = 0, Leptokurtisch > 0)')

---
## 2. Explorative Analyse (EDA)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Gold (GC=F) — Explorative Datenanalyse', fontsize=14, fontweight='bold')

axes[0].plot(close, color=GOLD, linewidth=1.0, label='Schlusskurs')
axes[0].fill_between(close.index, close, close.min(), alpha=0.1, color=GOLD)
axes[0].set_title('Schlusskurs (USD/oz)', fontweight='bold')
axes[0].set_ylabel('USD/oz')
events = [
    ('2020-03-18', 'COVID-Crash', 'red', -50),
    ('2020-08-06', 'COVID-ATH', 'green', 40),
    ('2022-03-08', 'Ukraine', 'orange', 40),
    ('2024-10-30', 'ATH 2024', 'gold', 40),
]
for date, label, color, yoff in events:
    idx = close.index.get_indexer([date], method='nearest')[0]
    axes[0].annotate(label, xy=(close.index[idx], close.iloc[idx]),
                     xytext=(0, yoff), textcoords='offset points',
                     fontsize=8, color=color, ha='center',
                     arrowprops=dict(arrowstyle='->', color=color, lw=1.0))
axes[0].legend()

axes[1].plot(log_ret, color=BLUE, linewidth=0.5, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].axhline(log_ret.mean() + 2*log_ret.std(), color=RED, linewidth=0.8,
                linestyle=':', label='±2σ')
axes[1].axhline(log_ret.mean() - 2*log_ret.std(), color=RED, linewidth=0.8, linestyle=':')
axes[1].set_title('Log-Returns (tägliche Renditen)', fontweight='bold')
axes[1].set_ylabel('log(Pt / Pt-1)')
axes[1].legend()

rolling_vol = log_ret.rolling(30).std() * np.sqrt(252) * 100
axes[2].plot(rolling_vol, color=RED, linewidth=1.0)
axes[2].fill_between(rolling_vol.index, rolling_vol, alpha=0.2, color=RED)
axes[2].axhline(rolling_vol.mean(), color='black', linewidth=0.8, linestyle='--',
                label=f'Ø Volatilität: {rolling_vol.mean():.1f}%')
axes[2].set_title('Rollierende 30-Tage-Volatilität (annualisiert)', fontweight='bold')
axes[2].set_ylabel('Volatilität (%)')
axes[2].legend()

plt.tight_layout()
os.makedirs('../reports', exist_ok=True)
plt.savefig('../reports/gold_eda.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Stationaritätsprüfung (Box-Jenkins Schritt 1)

In [ ]:
def adf_test(series, name=''):
    result = adfuller(series.dropna(), autolag='AIC')
    stat, pval, lags, nobs, crit = result[0], result[1], result[2], result[3], result[4]
    print(f'ADF-Test: {name}')
    print(f'  Teststatistik : {stat:.4f}')
    print(f'  p-Wert        : {pval:.4f}  → {"✅ Stationär" if pval < 0.05 else "❌ Nicht stationär"} (α=5%)')
    print(f'  Kritische Werte: 1%={crit["1%"]:.3f}, 5%={crit["5%"]:.3f}')
    print()

def kpss_test(series, name=''):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        stat, pval, lags, crit = kpss(series.dropna(), regression='c', nlags='auto')
    print(f'KPSS-Test: {name}')
    print(f'  Teststatistik : {stat:.4f}')
    print(f'  p-Wert        : {pval:.4f}  → {"❌ Nicht stationär" if pval < 0.05 else "✅ Stationär"} (α=5%)')
    print()

print('='*55)
print('STATIONARITÄTSPRÜFUNG — NIVEAUPREISE')
print('='*55)
adf_test(close, 'Gold Schlusskurs')
kpss_test(close, 'Gold Schlusskurs')

print('='*55)
print('STATIONARITÄTSPRÜFUNG — LOG-RETURNS')
print('='*55)
adf_test(log_ret, 'Gold Log-Returns')
kpss_test(log_ret, 'Gold Log-Returns')

print('→ Fazit: Niveaupreise sind I(1) → d=1 im ARIMA-Modell.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('ACF & PACF der Log-Returns — Gold', fontsize=13, fontweight='bold')

plot_acf(log_ret, lags=40, ax=axes[0], color=GOLD, vlines_kwargs={'colors': GOLD})
axes[0].set_title('Autokorrelationsfunktion (ACF)', fontweight='bold')
axes[0].set_xlabel('Lag')

plot_pacf(log_ret, lags=40, ax=axes[1], method='ywm', color=GOLD,
          vlines_kwargs={'colors': GOLD})
axes[1].set_title('Partielle ACF (PACF)', fontweight='bold')
axes[1].set_xlabel('Lag')

plt.tight_layout()
plt.savefig('../reports/gold_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

print('Interpretation:')
print('  ACF : Kein signifikanter Peak ab Lag 1 → wenig MA-Struktur')
print('  PACF: Kein signifikanter Peak ab Lag 1 → wenig AR-Struktur')
print('  → Random-Walk-Charakteristik der Goldpreise bestätigt')

---
## 4. Modellidentifikation & Schätzung (Box-Jenkins Schritt 2)

In [ ]:
TEST_SIZE  = 60
train      = close.iloc[:-TEST_SIZE]
test       = close.iloc[-TEST_SIZE:]
log_train  = log_close[:-TEST_SIZE]
test_vals  = close.values[-TEST_SIZE:]

print(f'Training: {train.index[0].date()} → {train.index[-1].date()} ({len(train)} Tage)')
print(f'Test    : {test.index[0].date()}  → {test.index[-1].date()}  ({len(test)} Tage)')

In [ ]:
# Gittersuche über ARIMA(p, 1, q)
results = []
for p in range(4):
    for q in range(4):
        try:
            model = ARIMA(log_train, order=(p, 1, q)).fit()
            fc    = np.exp(model.forecast(TEST_SIZE))
            mae   = np.mean(np.abs(fc - test_vals))
            rmse  = np.sqrt(np.mean((fc - test_vals)**2))
            mape  = np.mean(np.abs((fc - test_vals) / test_vals)) * 100
            mase  = mae / np.mean(np.abs(np.diff(train.values)))
            results.append({'Modell': f'ARIMA({p},1,{q})',
                            'p': p, 'd': 1, 'q': q,
                            'AIC': model.aic, 'BIC': model.bic,
                            'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape, 'MASE': mase})
        except:
            pass

res_df = pd.DataFrame(results).sort_values('AIC')
print('Top 8 Modelle nach AIC:')
print(res_df[['Modell','AIC','BIC','MAE','RMSE','MAPE (%)','MASE']].head(8).round(4).to_string(index=False))

In [ ]:
best_row = res_df.iloc[0]
p_best, d_best, q_best = int(best_row['p']), int(best_row['d']), int(best_row['q'])
print(f'✅ Bestes Modell: ARIMA({p_best},{d_best},{q_best})')

best_model = ARIMA(log_train, order=(p_best, d_best, q_best)).fit()

fc_obj    = best_model.get_forecast(TEST_SIZE)
fc_mean   = np.exp(fc_obj.predicted_mean)
fc_ci_raw = np.exp(fc_obj.conf_int(alpha=0.05))
fc_mean_s = pd.Series(fc_mean, index=test.index)
fc_ci     = pd.DataFrame(fc_ci_raw, columns=['KI_unten', 'KI_oben'], index=test.index)

mae  = np.mean(np.abs(fc_mean - test_vals))
rmse = np.sqrt(np.mean((fc_mean - test_vals)**2))
mape = np.mean(np.abs((fc_mean - test_vals) / test_vals)) * 100
mase = mae / np.mean(np.abs(np.diff(train.values)))

print(f'\nBacktesting-Metriken (60-Tage Out-of-Sample):')
print(f'  AIC   = {best_model.aic:.4f}')
print(f'  BIC   = {best_model.bic:.4f}')
print(f'  MAE   = {mae:.2f} USD/oz')
print(f'  RMSE  = {rmse:.2f} USD/oz')
print(f'  MAPE  = {mape:.4f}%')
print(f'  MASE  = {mase:.4f}')
print()
print(best_model.summary())

---
## 5. Residualdiagnostik (Box-Jenkins Schritt 3)

In [ ]:
residuals = pd.Series(best_model.resid[1:])  # ersten Tag entfernen

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(f'Residualdiagnostik — ARIMA({p_best},{d_best},{q_best})',
             fontsize=13, fontweight='bold')

axes[0, 0].plot(residuals.values, color=BLUE, linewidth=0.6, alpha=0.8)
axes[0, 0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0, 0].set_title('Residuenverlauf')
axes[0, 0].set_ylabel('Residuum')

axes[0, 1].hist(residuals, bins=60, density=True, color=GOLD, alpha=0.7, edgecolor='none')
x = np.linspace(residuals.min(), residuals.max(), 300)
axes[0, 1].plot(x, stats.norm.pdf(x, residuals.mean(), residuals.std()),
                color=RED, linewidth=2, label='Normalvert.')
axes[0, 1].set_title('Residuenverteilung')
axes[0, 1].legend()

plot_acf(residuals, lags=30, ax=axes[1, 0], color=BLUE,
         vlines_kwargs={'colors': BLUE})
axes[1, 0].set_title('ACF der Residuen')

stats.probplot(residuals, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q-Plot der Residuen')

plt.tight_layout()
plt.savefig('../reports/gold_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print('Ljung-Box-Test:')
lb = acorr_ljungbox(residuals, lags=[5, 10, 20], return_df=True)
lb.index = [f'Lag {l}' for l in [5, 10, 20]]
print(lb.round(4))
print('\n→ p > 0.05: Residuen sind Weißes Rauschen ✅' if (lb['lb_pvalue'] > 0.05).all() else
      '\n→ p < 0.05: Noch Autokorrelationsstruktur vorhanden ⚠️')

jb_stat, jb_p = stats.jarque_bera(residuals)
print(f'\nJarque-Bera-Test: Statistik={jb_stat:.2f}, p={jb_p:.4f}')
print('→ ' + ('Residuen normalverteilt ✅' if jb_p > 0.05 else
               'Residuen nicht normalverteilt (Leptokurtisch) — typisch für Finanzdaten ⚠️'))

---
## 6. Prognose (Box-Jenkins Schritt 4)

In [ ]:
final_model = ARIMA(log_close, order=(p_best, d_best, q_best)).fit()

fc10_obj   = final_model.get_forecast(steps=10)
fc10_mean  = np.exp(fc10_obj.predicted_mean)
fc10_ci    = np.exp(fc10_obj.conf_int(alpha=0.05))
fc10_dates = pd.bdate_range(close.index[-1] + pd.Timedelta(days=1), periods=10)

forecast_table = pd.DataFrame({
    'Handelstag': range(1, 11),
    'Datum': fc10_dates,
    'Prognose (USD/oz)': np.round(fc10_mean, 2),
    'KI untere Grenze':  np.round(fc10_ci[:, 0], 2),
    'KI obere Grenze':   np.round(fc10_ci[:, 1], 2),
    'KI-Breite':         np.round(fc10_ci[:, 1] - fc10_ci[:, 0], 2),
})
forecast_table.set_index('Handelstag', inplace=True)
print(f'10-Perioden-Prognose Gold — ARIMA({p_best},{d_best},{q_best}):')
print(forecast_table.to_string())

In [ ]:
fc10_mean_s = pd.Series(fc10_mean, index=fc10_dates)
fc10_ci_df  = pd.DataFrame(fc10_ci, columns=['KI_unten', 'KI_oben'], index=fc10_dates)

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle(f'Gold (GC=F) — ARIMA({p_best},{d_best},{q_best}) · Backtesting + 10-Tage-Prognose',
             fontsize=13, fontweight='bold')

ax.plot(train, color=GOLD, linewidth=1.2, label='Trainingsdaten')
ax.plot(test, color='gray', linewidth=1.2, label=f'Testdaten ({TEST_SIZE} Tage)')
ax.plot(fc_mean_s, color=BLUE, linewidth=1.5, linestyle='--', label='ARIMA-Prognose (Backtest)')
ax.fill_between(fc_ci.index, fc_ci['KI_unten'], fc_ci['KI_oben'],
                alpha=0.15, color=BLUE, label='95%-KI (Backtest)')
ax.plot(fc10_mean_s, color=GREEN, linewidth=2.0, marker='o', markersize=5,
        label='10-Tage-Prognose')
ax.fill_between(fc10_ci_df.index, fc10_ci_df['KI_unten'], fc10_ci_df['KI_oben'],
                alpha=0.25, color=GREEN, label='95%-KI (Prognose)')
ax.axvline(test.index[0], color=RED, linewidth=1.2, linestyle=':', alpha=0.8,
           label='Train/Test-Split')
ax.axvline(close.index[-1], color='purple', linewidth=1.2, linestyle=':',
           alpha=0.8, label='Prognosebeginn')
ax.set_xlim(train.index[0], fc10_mean_s.index[-1])

metric_text = f'MAE={mae:.2f} | RMSE={rmse:.2f} | MAPE={mape:.3f}% | MASE={mase:.3f}'
ax.text(0.01, 0.03, metric_text, transform=ax.transAxes, fontsize=9,
        color='dimgray', va='bottom',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

ax.set_ylabel('USD/oz', fontsize=11)
ax.set_xlabel('Datum', fontsize=11)
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/gold_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Modellvergleich & Ergebniszusammenfassung

In [ ]:
top5 = res_df.head(5)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Modellvergleich — Top 5 ARIMA-Modelle (Gold)', fontsize=13, fontweight='bold')

colors = [GOLD if i == 0 else 'steelblue' for i in range(len(top5))]
bars = axes[0].bar(top5['Modell'], top5['AIC'], color=colors, edgecolor='none', alpha=0.85)
axes[0].set_title('AIC (niedriger = besser)', fontweight='bold')
axes[0].set_ylabel('AIC')
axes[0].tick_params(axis='x', rotation=20)
axes[0].bar_label(bars, fmt='%.1f', fontsize=8, padding=3)

bars2 = axes[1].bar(top5['Modell'], top5['RMSE'], color=colors, edgecolor='none', alpha=0.85)
axes[1].set_title('RMSE (niedriger = besser)', fontweight='bold')
axes[1].set_ylabel('RMSE (USD/oz)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].bar_label(bars2, fmt='%.2f', fontsize=8, padding=3)

plt.tight_layout()
plt.savefig('../reports/gold_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"""
╔══════════════════════════════════════════════════════════════╗
║           ERGEBNISZUSAMMENFASSUNG — Gold (GC=F)             ║
╠══════════════════════════════════════════════════════════════╣
║  Bestes Modell : ARIMA({p_best},{d_best},{q_best})                               ║
║  AIC           : {best_model.aic:.2f}                           ║
║  BIC           : {best_model.bic:.2f}                           ║
╠══════════════════════════════════════════════════════════════╣
║  Backtesting (60-Tage Out-of-Sample):                       ║
║  MAE   = {mae:.2f} USD/oz                                  ║
║  RMSE  = {rmse:.2f} USD/oz                                  ║
║  MAPE  = {mape:.4f}%                                        ║
║  MASE  = {mase:.4f}                                         ║
╚══════════════════════════════════════════════════════════════╝
""")

In [ ]:
result_row = pd.DataFrame([{
    'Asset': 'Gold', 'Ticker': 'GC=F',
    'Modell': f'ARIMA({p_best},{d_best},{q_best})',
    'AIC': round(best_model.aic, 4), 'BIC': round(best_model.bic, 4),
    'MAE': round(mae, 4), 'RMSE': round(rmse, 4),
    'MAPE (%)': round(mape, 4), 'MASE': round(mase, 4),
}])

results_path = '../reports/results_cv.csv'
os.makedirs('../reports', exist_ok=True)
if os.path.exists(results_path):
    existing = pd.read_csv(results_path)
    existing = existing[existing['Asset'] != 'Gold']
    combined = pd.concat([existing, result_row], ignore_index=True)
else:
    combined = result_row

combined.to_csv(results_path, index=False)
print(f'Ergebnisse gespeichert: {results_path}')
print(combined)